# Setup

In [1]:
from pprint import pprint
import os
import numpy as np
import torch
from transformers.utils import logging
from transformers import set_seed

# Seed
seed = 42
set_seed(seed)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Suppress hf loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

W1019 15:00:32.073000 20696 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1019 15:00:32.146000 20696 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1019 15:00:33.130000 20696 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# 1. Chat Model

- Model List
  - `qwen3:0.6b`
  - `qwen3:1.7b`
  - `qwen3:4b`
  - `qwen3:8b`
  - `qwen3:14b`
  - `qwen3:32b`
  - `qwen3:4b-instruct`
  - `qwen3:4b-thinking`
  - `qwen3.5:4b`
  - `qwen3.5:9b`

## 1.1. Basic

In [2]:
from langchain_ollama import ChatOllama


# Model
chat = ChatOllama(
    model='qwen3:4b',
    temperature=1.0,
    top_k=5,
    top_p=0.9,
    seed=seed,
)

# Query
query = "What is 1 + 1?"

# Response
response = chat.invoke(query)

pprint(f"Response: {response.content}\n")

('Response: In standard arithmetic (the most common context for this '
 'question), **1 + 1 = 2**.  \n'
 '\n'
 'This is a fundamental rule of basic mathematics, where numbers represent '
 'quantities and the "+" symbol indicates addition.  \n'
 '\n'
 '### Why this answer?\n'
 '- **Arithmetic**: In the base-10 number system (which we use daily), adding '
 'two units (1) results in two units (2).  \n'
 '- **Binary**: If you meant binary (computer science), 1 + 1 = **10** (which '
 'equals 2 in decimal). But since the question didn’t specify a context, the '
 'default answer is **2**.  \n'
 '- **Real-world examples**:  \n'
 '  - If you have 1 apple and add another apple, you have **2 apples**.  \n'
 '  - If you have 1 dollar and add another dollar, you have **2 dollars**.  \n'
 '\n'
 '### Why not other answers?\n'
 '- **Philosophy/Abstract contexts?** (e.g., set theory, quantum mechanics) — '
 'These are *not* relevant here. The question is straightforward.  \n'
 '- **Language quirks?** (

## 1.2. Prompt Template

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


# Model
chat = ChatOllama(
    model='qwen3:4b',
    seed=seed,
)

# Template
system_prompt = """
You are a helpful assistant.
Answer precisely and shortly.
"""
query = "What is 2 * 3?"

template = ChatPromptTemplate.from_messages(
    [
        ('system', '{system_prompt}'),
        ('human', '{query}'),
    ]
)
prompt = template.invoke(
    {
        'system_prompt': system_prompt,
        'query': query,
    }
)
pprint(prompt.messages)

# Response
response = chat.invoke(prompt)

pprint(response.content)

[SystemMessage(content='\nYou are a helpful assistant.\nAnswer precisely and shortly.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 2 * 3?', additional_kwargs={}, response_metadata={})]
'6'


## 1.3. Reasoning

In [4]:
from langchain_ollama import ChatOllama


# Model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Query
query = "What is my name?"

# Response
response = chat.invoke(query)

pprint(f"Response: {response.content}\n")
pprint(f"Reasoning: {response.additional_kwargs['reasoning_content']}\n")

("Response: I don't have access to your personal information, so I don't know "
 'your name! 😊  \n'
 '\n'
 "As an AI, I don't have personal details about you—only what you share with "
 'me in our conversation. If you tell me your name, I can help you with '
 'anything you need!  \n'
 '\n'
 'What would you like to do together? 🌟\n')
('Reasoning: Okay, the user is asking "What is my name?" Hmm, this is '
 "interesting because I don't have personal information about them. I should "
 "clarify that I don't have access to their name or any personal details. \n"
 '\n'
 "I wonder why they're asking this. Maybe they're confused about how AI works, "
 "or they're testing if I can access their data. Could also be they're just "
 'being playful. \n'
 '\n'
 "I should respond clearly but gently. Important to emphasize that I don't "
 'have their name while keeping it friendly. No need to overcomplicate it - '
 'just state the facts simply. \n'
 '\n'
 'Also should offer help with other things since

## 1.4. Structured Output

In [5]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel


# Model
chat = ChatOllama(
    model='qwen3:4b',
    reasoninig=True,
    seed=seed,
)

# Structured Output
class AnswerWithJustification(BaseModel):
    answer: str
    justification: str

chat_structured = chat.with_structured_output(AnswerWithJustification)

# Response
response = chat_structured.invoke("What is 2 * 3?")

print(f"Answer: {response.answer}")
pprint(f"Justification: {response.justification}")


Answer: 6
('Justification: The multiplication of 2 and 3 is calculated as follows: 2 × 3 '
 '= 6. This can be verified by adding 2 three times (2 + 2 + 2 = 6) or 3 two '
 'times (3 + 3 = 6).')


## 1.5. Streaming

In [6]:
from langchain_ollama import ChatOllama
from time import sleep


# Model
chat = ChatOllama(
    model='qwen3:4b',
    reasoninig=True,
    seed=seed,
)

# Response
response = chat.stream("What is 2 * 3?")

for part in response:
    if part.content != '':
        print(part.content, end="", flush=True)
        sleep(0.1)

The result of **2 * 3** is **6**.  

This is because multiplication represents repeated addition:  
- 2 + 2 + 2 = 6 (adding 2 three times)  
- or 3 + 3 = 6 (adding 3 two times).  

It's a basic arithmetic operation with no ambiguity. 😊

# 2. Component Integration

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain


# Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Template
system_prompt = """
You are a helpful assistant.
Answer concisely and shortly.
"""

template = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', '{query}'),
    ],
)

# Chatbot = template + chat
## Method 1. Chain
@chain
def chatbot(values):
    prompt = template.invoke(values)
    response = chat.invoke(prompt)

    return response

## Method 2. LCEL: LangChain Expression Language
chatbot = template | chat

# Response
message = {'query': "What is 1 + 1?"}
response = chatbot.invoke(message)

pprint(response.content)

'2'


# 3. Tool Use

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from typing import List


# Template
system_prompt = """
You are a helpful assistant.
Answer concisely and shortly.
"""

template = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', '{query}'),
    ],
)

# Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Tool
@tool
def add_ints(x: int, y: int) -> int:
    """Add two integers."""
    return x + y

@tool
def sub_ints(x: int, y: int) -> int:
    """Substitute two integers."""
    return x - y

# Response format
class AnswerWithTools(BaseModel):
    answer: str
    tools: List[str]

# Agent
agent = create_agent(
    model=chat,
    tools=[add_ints, sub_ints],   # tool binding.
    response_format=AnswerWithTools,
)

# Chatbot = template + agent
chatbot = template | agent

# Response
response = chatbot.invoke({'query': "What is 1 + 1?"})

pprint(f"Tool calls: {response['messages'][2].tool_calls}")
print(f"Response: {response['messages'][-1].content}")
print(f"Structured response: {response['structured_response']}")

response_json = response['structured_response'].model_dump_json()

response_json

("Tool calls: [{'name': 'add_ints', 'args': {'x': 1, 'y': 1}, 'id': "
 "'a523a6fa-6a09-446a-8109-98432b5fd113', 'type': 'tool_call'}]")
Response: Returning structured response: answer='2' tools=['add_ints']
Structured response: answer='2' tools=['add_ints']


'{"answer":"2","tools":["add_ints"]}'